In [1]:
import os
import re
import json
import pandas as pd

# =========================
# ABSOLUTE INPUT PATHS
# =========================
DF_PATH = "/media/ausaafkhan/New Volume1/project_exhibition/broken_dataset_repair/data/real/df_working.csv"
PROFILE_PATH = "/media/ausaafkhan/New Volume1/project_exhibition/broken_dataset_repair/reports/profiling_reports/dataset_profile.csv"

# =========================
# OUTPUT PATHS
# =========================
OUTPUT_DIR = "/media/ausaafkhan/New Volume1/project_exhibition/broken_dataset_repair/reports/semantic_reports"
CSV_OUTPUT_PATH = os.path.join(OUTPUT_DIR, "semantic_types.csv")
JSON_OUTPUT_PATH = os.path.join(OUTPUT_DIR, "semantic_types.json")

# =========================
# FAIL-FAST: INPUT FILE CHECK
# =========================
if not os.path.exists(DF_PATH):
    raise FileNotFoundError(f"df_working not found at {DF_PATH}")

if not os.path.exists(PROFILE_PATH):
    raise FileNotFoundError(f"dataset_profile not found at {PROFILE_PATH}")

# Ensure output directory exists
os.makedirs(OUTPUT_DIR, exist_ok=True)

In [2]:
df_working = pd.read_csv(DF_PATH)
dataset_profile = pd.read_csv(PROFILE_PATH)

# =========================
# FAIL-FAST: COLUMN CONSISTENCY
# =========================
df_cols = set(df_working.columns)
profile_cols = set(dataset_profile["column_name"])

if df_cols != profile_cols:
    raise ValueError("Mismatch between df_working columns and dataset_profile columns")

In [3]:
def is_boolean(series, unique_values):
    valid_sets = [
        {"0", "1"},
        {"true", "false"},
        {"yes", "no"},
    ]
    
    normalized = set(str(v).strip().lower() for v in unique_values if pd.notnull(v))
    
    for valid in valid_sets:
        if normalized.issubset(valid) and len(normalized) == 2:
            return True, f"Values match boolean set {valid}"
    
    return False, ""


def is_datetime(series):
    patterns = [
        r"^\d{4}-\d{2}-\d{2}$",        # YYYY-MM-DD
        r"^\d{2}/\d{2}/\d{4}$",        # DD/MM/YYYY
        r"^\d{4}-\d{2}-\d{2}T\d{2}:\d{2}:\d{2}$"  # ISO
    ]
    
    non_null = series.dropna().astype(str)
    
    if len(non_null) == 0:
        return False, ""
    
    for pattern in patterns:
        if non_null.str.match(pattern).all():
            return True, f"All values match datetime regex: {pattern}"
    
    return False, ""


def is_id(unique_ratio, dtype):
    if unique_ratio > 0.95:
        return True, f"High uniqueness ratio: {unique_ratio}"
    return False, ""


def is_numeric_continuous(series, dtype, unique_count):
    if "float" in dtype:
        return True, "Float dtype indicates continuous values"
    
    if unique_count > 20 and series.dropna().astype(str).str.contains(r"\.").any():
        return True, "Decimal presence + high cardinality"
    
    return False, ""


def is_numeric_discrete(series, dtype, unique_count):
    if "int" in dtype and unique_count <= 20:
        return True, "Integer dtype with low cardinality"
    
    if "int" in dtype:
        return True, "Integer dtype indicates discrete values"
    
    return False, ""


def is_categorical(unique_ratio, duplicate_percentage):
    if unique_ratio < 0.2 and duplicate_percentage > 50:
        return True, f"Low uniqueness ({unique_ratio}) + high duplication ({duplicate_percentage})"
    
    return False, ""


def is_text(series, avg_length, unique_ratio):
    if avg_length > 15 and unique_ratio > 0.5:
        return True, f"High avg length ({avg_length}) + high uniqueness ({unique_ratio})"
    
    return False, ""

In [4]:
semantic_records = []

total_rows = len(df_working)

for _, row in dataset_profile.iterrows():
    
    col = row["column_name"]
    dtype = str(row["dtype"]).lower()
    unique_count = int(row["unique_count"])
    duplicate_percentage = float(row["duplicate_percentage"])
    null_percentage = float(row["null_percentage"])
    
    series = df_working[col]
    
    unique_ratio = unique_count / total_rows if total_rows > 0 else 0
    
    avg_length = series.dropna().astype(str).str.len().mean() if series.notna().any() else 0
    
    unique_values = series.dropna().unique()
    
    semantic_type = None
    reason = ""
    confidence = 0.0

    # =========================
    # RULE ORDER (STRICT)
    # =========================
    
    # BOOLEAN
    flag, msg = is_boolean(series, unique_values)
    if flag:
        semantic_type = "boolean"
        reason = f"BOOLEAN: {msg}"
        confidence = 1.0
    
    # DATETIME
    if semantic_type is None:
        flag, msg = is_datetime(series)
        if flag:
            semantic_type = "datetime"
            reason = f"DATETIME: {msg}"
            confidence = 1.0
    
    # ID
    if semantic_type is None:
        flag, msg = is_id(unique_ratio, dtype)
        if flag:
            semantic_type = "id"
            reason = f"ID: {msg}"
            confidence = 0.95
    
    # NUMERIC CONTINUOUS
    if semantic_type is None:
        flag, msg = is_numeric_continuous(series, dtype, unique_count)
        if flag:
            semantic_type = "numeric_continuous"
            reason = f"NUM_CONT: {msg}"
            confidence = 0.9
    
    # NUMERIC DISCRETE
    if semantic_type is None:
        flag, msg = is_numeric_discrete(series, dtype, unique_count)
        if flag:
            semantic_type = "numeric_discrete"
            reason = f"NUM_DISC: {msg}"
            confidence = 0.85
    
    # CATEGORICAL
    if semantic_type is None:
        flag, msg = is_categorical(unique_ratio, duplicate_percentage)
        if flag:
            semantic_type = "categorical"
            reason = f"CATEGORICAL: {msg}"
            confidence = 0.9
    
    # TEXT
    if semantic_type is None:
        flag, msg = is_text(series, avg_length, unique_ratio)
        if flag:
            semantic_type = "text"
            reason = f"TEXT: {msg}"
            confidence = 0.85
    
    # UNKNOWN
    if semantic_type is None:
        semantic_type = "unknown"
        reason = "No deterministic rule matched"
        confidence = 0.5

    semantic_records.append({
        "column_name": col,
        "semantic_type": semantic_type,
        "confidence_score": confidence,
        "inference_reason": reason
    })

In [5]:
semantic_types = pd.DataFrame(semantic_records)

# =========================
# VALIDATIONS
# =========================

# Row count check
if len(semantic_types) != len(df_working.columns):
    raise ValueError("Row count mismatch in semantic_types")

# No missing types
if semantic_types["semantic_type"].isnull().any():
    raise ValueError("Missing semantic_type detected")

# Confidence bounds
if not semantic_types["confidence_score"].between(0, 1).all():
    raise ValueError("Confidence scores out of bounds")

# Schema check
expected_cols = [
    "column_name",
    "semantic_type",
    "confidence_score",
    "inference_reason"
]

if list(semantic_types.columns) != expected_cols:
    raise ValueError("Schema mismatch in semantic_types")

In [6]:
semantic_types.to_csv(CSV_OUTPUT_PATH, index=False)

with open(JSON_OUTPUT_PATH, "w") as f:
    json.dump(semantic_records, f, indent=4)

print("Semantic type inference completed successfully.")

Semantic type inference completed successfully.
